In [ ]:
import os
import re
import numpy as np
import scipy as sp
import pickle
# import matplotlib.pyplot as plt
# import pandas as pd
HP_FEATURES = 60
EPSILON  = 1e-10
NUM_OF_NEGATIVE_COUPLES = 6
LR = 0.01 # LEARNING_RATE
WINDOW = 5

In [2]:
assert WINDOW%2 == 1
center_window_index = int(np.ceil(WINDOW/2)) - 1

context_indeces = list(range(WINDOW))
del(context_indeces[int(center_window_index)])

In [3]:
with open(os.path.join('data','text8'), mode='r', encoding='utf-8') as fw:
    data = fw.read()

cleaned_data = re.split(r'\W+', data)
lex_from_num_to_tokens = dict(enumerate(sorted(set( cleaned_data ))))
lex_from_tokens_to_num = dict()
for k,v in lex_from_num_to_tokens.items():
    lex_from_tokens_to_num[v] = k

distinct_num = len(lex_from_num_to_tokens)


indexed_data = []
for token in cleaned_data:
    indexed_data.append(lex_from_tokens_to_num[token])

center_embeddings = (np.random.rand(distinct_num,HP_FEATURES)-0.5)
context_embeddings = (np.random.rand(distinct_num,HP_FEATURES)-0.5)

In [4]:
from collections import Counter
sampling_weights = Counter(indexed_data)
noise_distribution = dict()
for id,num_of_occurrences in sampling_weights.items():
    noise_distribution[id] = num_of_occurrences**(3/4)

denominator = sum(noise_distribution.values())
lexicon = []
relative_frequencies = []
for id, freq in noise_distribution.items():
    noise_distribution[id] = freq/denominator
    lexicon.append(id)
    relative_frequencies.append(freq/denominator)


KeyboardInterrupt: 

In [ ]:
def check(w1, w2):
    c = center_embeddings[lex_from_tokens_to_num[w1]]
    x = context_embeddings[lex_from_tokens_to_num[w2]]
    return sp.special.expit(c.dot(x))


In [ ]:

correlation_expected = [("of", "the"), ("in", "the"), ("and", "the"), ("to", "the"),
("as", "a"), ("is", "a"), ("was", "a"), ("state", "government"),
("political", "party"), ("anarchism", "anarchist"), ("war", "military"),
("century", "history"), ("people", "society"), ("law", "government"),
("revolution", "socialist"), ("world", "history"), ("economic", "political"),
("power", "control"), ("society", "culture"), ("early", "first")]

correlation_not_expected = [("anarchism", "banana"), ("government", "spaghetti"), ("war", "butterfly"),
("century", "guitar"), ("political", "sandwich"), ("society", "bicycle"),
("history", "pillow"), ("revolution", "umbrella"), ("economic", "dolphin"),
("power", "lettuce"), ("law", "balloon"), ("state", "pancake"),
("people", "telescope"), ("world", "spoon"), ("movement", "cactus"),
("socialist", "giraffe"), ("military", "cupcake"), ("culture", "hammer"),
("control", "jellyfish"), ("party", "volcano")]

for couple in correlation_expected:
    print(f"{couple[0]} * {couple[1]} = {check(couple[0], couple[1])}")
print('=================================================')
for couple in correlation_not_expected:
    print(f"{couple[0]} * {couple[1]} = {check(couple[0], couple[1])}")

### For positive couples
$G_{positive} = loss(c,x) = loss(sigmoid(dot\_product(c,x))) = -ln(\frac{1}{1+e^{-c \cdot x}}) \in \R$

$G_{positive, center}' = \frac{\partial loss}{\partial c}(loss(c,x)) = [sigmoid(dot\_product(c,x))- 1] * x = [sigmoid(c \cdot x)- 1] * x \in \R^n$

$G_{positive, context}' = \frac{\partial loss}{\partial x}(loss(c,x)) = [sigmoid(dot\_product(c,x))- 1] * c = [sigmoid(c \cdot x)- 1] * c \in \R^n$

### For negative couples
$G_{negative} = loss(c,x) = loss(sigmoid(dot\_product(c,x))) = -ln(1 - \frac{1}{1+e^{-c \cdot x}}) \in \R$

$G_{negative, center}' = \frac{\partial loss}{\partial c}(loss(c,x)) = sigmoid(dot\_product(c,x)) * x = sigmoid(c \cdot x) * x \in \R^n$

$G_{negative, context}' = \frac{\partial loss}{\partial x}(loss(c,x)) = sigmoid(dot\_product(c,x)) * c = sigmoid(c \cdot x) * c \in \R^n$

### gradient descent
$LR = 0.01$ Learning Rate (hyperparameter)

#### For positive couples
$center = center - LR * G_{positive, center}' = center - LR * [sigmoid(c \cdot x)- 1] * x \in \R^n$

$context = context - LR * G_{positive, context}' = context - LR * [sigmoid(c \cdot x)- 1] * c \in \R^n$

#### For negative couples
$center = center - LR * G_{negative, center}' = center - LR * sigmoid(c \cdot x) * x \in \R^n$

$context = context - LR * G_{negative, context}' = context - LR * sigmoid(c \cdot x) * c \in \R^n$




In [ ]:
iii = 1_000
for ind in range(len(indexed_data[:300_000])-WINDOW): # the :10 is only for test in order not to run the full loop while I am testing the code

    if ind == iii:
        print(ind)
        iii+=1_000
    # print(indexed_data[ind+center_window_index])
    center = center_embeddings[indexed_data[ind+center_window_index]]

    #positive couples
    positive_couples_dot_products = [center.dot(context_embeddings[indexed_data[ind+con]]) for con in context_indeces ]

    # negative couples
    negative_couples_dot_products = []
    random_context_indeces = [np.random.choice(lexicon, p=relative_frequencies) for i in range(NUM_OF_NEGATIVE_COUPLES)]
    negative_couples_dot_products = [center.dot(context_embeddings[i]) for i in random_context_indeces]


    for con_index, positive_couples_dot_product in zip(context_indeces,positive_couples_dot_products):
        sigmoid = sp.special.expit(positive_couples_dot_product)
        center_embeddings[indexed_data[ind+center_window_index]] -=  LR * (sigmoid - 1.0) * context_embeddings[indexed_data[ind+con_index]]
        context_embeddings[indexed_data[ind+con_index]] -= LR * (sigmoid - 1.0) * center 

    for random_con_index, negative_couples_dot_product in zip(random_context_indeces,negative_couples_dot_products):
        sigmoid = sp.special.expit(negative_couples_dot_product)
        center_embeddings[indexed_data[ind+center_window_index]] -=  LR * sigmoid * context_embeddings[random_con_index]
        context_embeddings[random_con_index] -= LR * sigmoid * center

In [ ]:
np.save("center_embeddings.npy", center_embeddings)
np.save("context_embeddings.npy", context_embeddings)

mydata = {
    "lex_from_num_to_tokens":lex_from_num_to_tokens,
    "lex_from_tokens_to_num":lex_from_tokens_to_num,
    "indexed_data":indexed_data
}

import pickle
with open("values.bin", "wb") as f:
    pickle.dump(mydata, f)


In [ ]:
# sigmoid = sp.special.expit(dot_product)
# print(sigmoid)
# - np.log(   1-sp.special.expit(center.dot(con_0)) + EPSILON     )
# - np.log(    sp.special.expit(center.dot(context_embeddings[negative_sample])) + EPSILON     )

In [ ]:
# center_embeddings = np.load("center_embeddings.npy")
# context_embeddings = np.load("context_embeddings.npy")

# with open("values.bin", "rb") as f:
#     mydata = pickle.load(f)

# lex_from_num_to_tokens = mydata["lex_from_num_to_tokens"]
# lex_from_tokens_to_num = mydata["lex_from_tokens_to_num"]
# indexed_data = mydata["indexed_data"]

In [ ]:

for couple in correlation_expected:
    print(f"{couple[0]} * {couple[1]} = {check(couple[0], couple[1])}")
print('=================================================')
for couple in correlation_not_expected:
    print(f"{couple[0]} * {couple[1]} = {check(couple[0], couple[1])}")